# 04 · Klasik kıyas: 2RC eşdeğer devre + EKF
Tanımlama yalnız eğitim dosyalarından (C1–C4), sıcaklık başına; R(T) üstel uyum, OCV ve τ sıcaklıkta ara değerleme; −20°C ekstrapolasyon.
İki başlangıç: **doğru SoC0** (EKF lehine) ve **hatalı SoC0 (−0.2)**. Metrikler derin modellerle aynı indekslerde (t ≥ W) hesaplanır.

In [ ]:
import sys, os, json, time, warnings
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)
SMOKE = False                 # True: çok küçük hızlı deneme (kurulum testi)  False
sys.path.insert(0, ROOT)
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from data_utils import *
from models.experiment import *
from models import engine
exp = Experiment(ROOT, smoke=SMOKE)
exp.load_data()
print('Cihaz:', exp.dev, '| W =', exp.cfg['W'])

In [ ]:
from models.ekf import fit_temperature_model, ECMParams, run_ekf
temps_c = {'25degC': 25.0, '10degC': 10.0, '0degC': 0.0, '-10degC': -10.0}
t0 = time.time(); tm = fit_temperature_model(exp.role_dfs, TRAIN_ROLES, temps_c); t_id = time.time() - t0
json.dump(tm, open(exp.p['configs'] / 'ecm_parametreleri.json', 'w', encoding='utf-8'), indent=1, default=float)
P = ECMParams(tm)
ptab = pd.DataFrame([{'temp': t, **{k: v for k, v in p.items() if k != 'ocv_coef'}} for t, p in tm['per_temp'].items()])
print(f'Tanımlama süresi {t_id:.1f} s'); print(ptab.round(4).to_string(index=False))

In [ ]:
W = exp.cfg['W']; rows, sers = [], []
for init, dsoc in [('doğru SoC0', 0.0), ('hatalı SoC0 (−0.2)', 0.2)]:
    for (t, c), d in exp.te_dfs.items():
        t0 = time.perf_counter(); est = run_ekf(d, P, d.soc.iloc[0] - dsoc); ms = (time.perf_counter() - t0) / len(d) * 1e3
        e, tr = est[W:], d.soc.values[W:]
        m = engine.metrics(e, tr); err = e - tr
        rows.append({'model': '2RC-ECM + EKF', 'başlangıç': init, 'seed': 0, 'temp': t, 'cycle': c, **m, **{f'{k}_ema': v for k, v in m.items()},
                     **engine.decompose(err), 'ms_adım': ms})
        if dsoc == 0: sers.append(pd.DataFrame({'temp': t, 'cycle': c, 't_s': d.time.values[W:] - d.attrs['time0'], 'soc_true': tr, 'soc_pred': e, 'err': err}))
ekf = pd.DataFrame(rows)
(exp.p['series'] / 'ekf').mkdir(exist_ok=True)
save_table(pd.concat(sers), exp.p['series'] / 'ekf' / 'ekf_dogru_soc0.parquet')
ekf.to_csv(exp.p['excel'] / '04_ekf_satirlar.csv', index=False)
t = (ekf.assign(k=np.where(ekf.temp == '-20degC', '−20°C', 'eğitim sıc.')).groupby(['başlangıç', 'k']).rmse.mean() * 100).unstack('k')
print(t.round(3).to_string()); print('Adım başına süre (ms):', ekf.ms_adım.mean().round(4))
with pd.ExcelWriter(exp.p['excel'] / '04_ekf.xlsx') as xw:
    ptab.to_excel(xw, sheet_name='ECM parametreleri', index=False); ekf.to_excel(xw, sheet_name='Sonuçlar', index=False)